In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Medallion Pattern: Silver Layer Pipeline
# MAGIC **Processes:** Bronze → Silver (Clean, Validate, Type-Cast, De-dup, Quarantine)

In [0]:
import re
import logging
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, DoubleType, IntegerType, DateType
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from setup import load_config, get_all_tables,get_raw_path
from src.utils.schemas import ORDERS_SCHEMA,CUSTOMER_SCHEMA,PRODUCTS_SCHEMA
import pyspark.sql.functions as F

In [0]:
config = load_config()

# unpack into a dict — same names as your original constants
tables = get_all_tables(config)
path=get_raw_path(config)

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("Cleaning layer")

 
# ── Volume path for raw uploaded files ───────────────────────────────────────
# Files are uploaded via UI to: Catalog > retail_catalog > retail > raw_files
VOLUME_RAW = path
 
RAW_PRODUCTS  = f"{VOLUME_RAW}/Products.csv"
RAW_CUSTOMERS = f"{VOLUME_RAW}/Customer.xlsx"
RAW_ORDERS    = f"{VOLUME_RAW}/Orders.json"

BRONZE_PRODUCTS  = tables['BRONZE_PRODUCTS']
BRONZE_CUSTOMERS = tables['BRONZE_CUSTOMERS']
BRONZE_ORDERS    = tables['BRONZE_ORDERS']
 
SILVER_PRODUCTS  = tables['SILVER_PRODUCTS']
SILVER_CUSTOMERS = tables['SILVER_CUSTOMERS']
SILVER_ORDERS    = tables['SILVER_ORDERS']
 
GOLD_SALES       = tables["GOLD_SALES"]
ENRICHED_ORDERS =  tables["GOLD_ENRICHED"]

In [0]:
def write_silver(df: DataFrame, target: str) -> None:
    """Upsert into silver Delta table; create on first run."""
    spark.catalog.tableExists(target)      # warm up catalog
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(target)

In [0]:
def clean_customers() -> DataFrame:
    df_customer = spark.table(BRONZE_CUSTOMERS)
    df_customer = df_customer.dropna(how="all")

    # Log rows with null or empty customer_name
    bad_name = df_customer.filter(
        F.col("customer_name").isNull() | (F.trim(F.col("customer_name")) == "")
    )
    # if bad_name.count() > 0:
    #     log_quarantine(bad_name, "bronze.customers", "null_customer_name")

    df_customer = df_customer.withColumn(
        "customer_name",
        F.when(
            F.col("customer_name").isNull() | (F.trim(F.col("customer_name")) == ""),
            F.lit("Unknown")
        ).otherwise(F.col("customer_name"))
    )

    df_customer = df_customer.withColumn("customer_name_raw", F.col("customer_name"))

    # Strip everything except letters, space, hyphen, apostrophe, dot
    df_customer = df_customer.withColumn(
        "customer_name",
        F.regexp_replace(F.col("customer_name"), r"[^a-zA-Z .\-']", "")
    )
    df_customer = df_customer.withColumn(
        "customer_name",
        F.regexp_replace(F.col("customer_name"), r"(?<![a-zA-Z])-|-(?![a-zA-Z])", "")
    )
    df_customer = df_customer.withColumn(
        "customer_name",
        F.regexp_replace(F.col("customer_name"), r"(?<![a-zA-Z])'|'(?![a-zA-Z])", "")
    )
    df_customer = df_customer.withColumn(
        "customer_name",
        F.regexp_replace(F.col("customer_name"), r"\.(?!\s|$)", "")
    )
    # Collapse multiple spaces and trim edges
    df_customer = df_customer.withColumn(
        "customer_name",
        F.trim(F.regexp_replace(F.col("customer_name"), r" {2,}", " "))
    )

    # Clean email
    df_customer = df_customer.withColumn("email", F.lower(F.trim(F.col("email"))))

    # Postal code – zero-pad to 5 digits (US)
    df_customer = df_customer.withColumn(
        "postal_code",
        F.when(
            F.length(F.col("postal_code").cast(StringType())) < 5,
            F.lpad(F.col("postal_code").cast(StringType()), 5, "0")
        ).otherwise(F.col("postal_code").cast(StringType()))
    )

    # Preserve raw phone
    df_customer = df_customer.withColumn("phone_raw", F.col("phone"))

    # Step 1 — nullify corrupt: #ERROR! and bare negatives
    df_customer = df_customer.withColumn(
        "phone",
        F.when(F.col("phone").rlike(r"^#ERROR!$|^-\d+$"), F.lit(None))
         .otherwise(F.col("phone"))
    )

    # Step 2 — extract extension into its own column before stripping
    df_customer = df_customer.withColumn(
        "phone_extension",
        F.when(
            F.col("phone").rlike(r"(?i)x\d+$"),
            F.regexp_extract(F.col("phone"), r"(?i)x(\d+)$", 1)
        ).otherwise(F.lit(None))
    )

    # Step 3 — strip extension from base
    df_customer = df_customer.withColumn(
        "phone",
        F.regexp_replace(F.col("phone"), r"(?i)x\d+$", "")
    )

    # Step 4 — digits only
    df_customer = df_customer.withColumn(
        "phone",
        F.regexp_replace(F.col("phone"), r"\D", "")
    )

    # Step 5 — strip international exit code 00 then country code 1 if 11 digits
    df_customer = df_customer.withColumn(
        "phone",
        F.when(F.col("phone").rlike(r"^00"), F.regexp_replace(F.col("phone"), r"^00", ""))
         .otherwise(F.col("phone"))
    )
    df_customer = df_customer.withColumn(
        "phone",
        F.when(
            (F.length(F.col("phone")) == 11) & F.col("phone").rlike(r"^1"),
            F.substring(F.col("phone"), 2, 10)
        ).otherwise(F.col("phone"))
    )

    # Step 6 — must be exactly 10 digits
    df_customer = df_customer.withColumn(
        "phone",
        F.when(F.length(F.col("phone")) == 10, F.col("phone"))
         .otherwise(F.lit(None))
    )

    # Segment – title-case & validate allowed values
    allowed_segments = ["Consumer", "Corporate", "Home Office"]
    df_customer = df_customer.withColumn("segment", F.initcap(F.col("segment")))
    df_customer = df_customer.filter(F.col("segment").isin(allowed_segments))

   
    
    try:
        write_silver(df_customer, SILVER_CUSTOMERS)
        print(f"Table load completed for table {SILVER_CUSTOMERS}")
    except:
        print(f"Table load failed for table {SILVER_CUSTOMERS}")
        
    return df_customer

In [0]:
def clean_products() -> DataFrame:
    df_products = spark.table(BRONZE_PRODUCTS)
    
    # ── 2a. clean product name column ────────────────────────────────────────
    df_products = (df_products
    .withColumn("product_name", F.regexp_replace("product_name", "\u00a0", " "))
    .withColumn("product_name", F.trim(F.col("product_name")))
    .withColumn("product_name", F.regexp_replace("product_name", " {2,}", " "))
    .withColumn("product_name", F.regexp_replace("product_name", r"[.,]+$", ""))
    .withColumn("product_name", F.initcap(F.col("product_name")))
)
    
    # ── 2b. price_per_product: must be positive ──────────────────────────────
   
    df_products = df_products.filter(F.col("price_per_product") > 0)

    #dedup
    window_spec = Window.partitionBy("product_id").orderBy(F.col("price_per_product"))
    df_products_deduped = (
            df_products.withColumn("rn", F.row_number().over(window_spec))
              .filter(F.col("rn") == 1)
              .drop("rn", "state")           # state becomes ambiguous after dedup
        )
    
    # ── 2d. Category & sub_category – title-case ────────────────────────────
    df_products_deduped = df_products_deduped.withColumn("category", F.initcap(F.col("category")))
    df_products_deduped = df_products_deduped.withColumn("sub_category", F.initcap(F.col("sub_category")))
    
    try:
        write_silver(df_products_deduped, SILVER_PRODUCTS)
        print(f"Table load completed for table {SILVER_PRODUCTS}")
    except:
        print(f"Table load failed for table {SILVER_PRODUCTS}")
    return df_products_deduped

In [0]:
def clean_orders() -> DataFrame:
    df_orders = spark.table(BRONZE_ORDERS)
    df_orders = df_orders.withColumn("order_date", F.to_date(F.col("order_date"), "d/M/yyyy"))
    df_orders = df_orders.withColumn("ship_date",  F.to_date(F.col("ship_date"),  "d/M/yyyy"))

    # ── 3c. Ship date must not precede order date ────────────────────────────
    
    df_orders = df_orders.filter(F.col("ship_date") >= F.col("order_date"))
    # ── 3d. Quantity must be ≥ 1 ─────────────────────────────────────────────
   
    df_orders = df_orders.filter(F.col("quantity") > 0)

    # ── 3e. Price must be positive ───────────────────────────────────────────
    df_orders = df_orders.filter(F.col("price") > 0)
    
    # ── 3f. Discount must be in [0, 1] ───────────────────────────────────────
    df_orders = df_orders.filter((F.col("discount") >= 0) & (F.col("discount") <= 1))

    silver_products = spark.table(SILVER_PRODUCTS).select("product_id")
    orphan_products = df_orders.join(silver_products, on="product_id", how="left_anti")
    df_orders = df_orders.join(silver_products, on="product_id", how="inner")

    # ── 3h. Derived columns ──────────────────────────────────────────────────
    df_orders = df_orders.withColumn("days_to_ship", F.datediff(F.col("ship_date"), F.col("order_date")))
    df_orders = df_orders.withColumn("net_revenue", F.round(F.col("price") * (1 - F.col("discount")), 2))
    df_orders = df_orders.withColumn("profit_margin",
            F.when(F.col("price") > 0, F.round(F.col("profit") / F.col("price"), 4)).otherwise(None)
        )
    try:
        write_silver(df_orders, SILVER_ORDERS)
        print(f"Table load completed for table{SILVER_ORDERS}")
    except:
        print(f"Table load failed for table {SILVER_ORDERS}")
    return df_orders

In [0]:
def enrich_orders() -> DataFrame:
    orders = spark.table(SILVER_ORDERS)
    customers = spark.table(SILVER_CUSTOMERS).select(
            F.col("customer_id").alias("cust_id"), "customer_name", "Country"
        )
    products  = spark.table(SILVER_PRODUCTS).select(
           F.col("product_id").alias("prod_id"), "product_name", "category", "sub_category"
        )
    
    df_enriched_orders = (
            orders
            .join(customers, orders.customer_id == customers.cust_id, how="left")
            .join(products,  orders.product_id == products.prod_id,  how="left")
        )
    df_enriched_orders = df_enriched_orders.withColumn("profit", F.round(F.col("profit"), 2))
    df_enriched_orders = df_enriched_orders.select("order_id", "order_date", "customer_id", "ship_date", "ship_mode", "days_to_ship", "quantity", "price", "discount", "profit","net_revenue", "profit_margin", "customer_name", "Country","product_id","product_name", "category","sub_category")

    try:
        write_silver(df_enriched_orders, ENRICHED_ORDERS)
        print(f"Table load completed for table {ENRICHED_ORDERS}")
    except:
        print(f"Table load failed for table {ENRICHED_ORDERS}")
    return df_enriched_orders

In [0]:
if __name__ == "__main__":
    clean_customers()
    clean_products()
    clean_orders()
    enrich_orders()
    print("\nSilver layer build complete.")